# Imports

In [ ]:
from pathlib import Path
import pandas as pd
from diive.core.io.files import save_parquet, load_parquet
from datetime import datetime

# Load main data

In [ ]:
SOURCEDIR = r"../40_FLUX_PROCESSING_CHAIN"
FILENAME = r"46.1_FLUXES_L3.2_NEE_LE_H_FN2O_FCH4.parquet"
FILEPATH = Path(SOURCEDIR) / FILENAME
maindf = load_parquet(filepath=FILEPATH)
maindf


# Load REddyProc output

In [ ]:
SOURCEDIR = r"../50_REDDYPROC"
FILENAME = r"51.1_UstarThresholds_MDS-gapfilled_NEE-partitioned.csv"
FILEPATH = Path(SOURCEDIR) / FILENAME
reddy = pd.read_csv(FILEPATH)

# Convert timestamp to datetime and change from end to middle
reddy['TIMESTAMP_MIDDLE'] = pd.to_datetime(reddy['TIMESTAMP_END'], format='%Y-%m-%dT%H:%M:%SZ', errors='coerce') - pd.Timedelta(minutes=15)
# Drop the timestamp end
reddy.drop(columns='TIMESTAMP_END', inplace=True)
# Set the timestamp as index
reddy.set_index('TIMESTAMP_MIDDLE', inplace=True)

# Add a string to identify the reddyproc columns later
reddy.columns = [f'{c}_reddyproc' for c in reddy.columns]

reddy

# Merge

In [ ]:
newcols = [c for c in reddy.columns if c not in maindf]
print("NEW VARIABLES FROM REDDYPROC:")
[print(f"+ {c}") for c in newcols]
maindf = pd.concat([maindf, reddy[newcols]], axis=1)
maindf

# Export 

In [ ]:
filename = "61.1_FLUXES_L3.2_NEE_LE_H_FN2O_FCH4_REDDYPROC"
save_parquet(data=maindf, filename=filename)

# End of notebook

In [ ]:
dt_string = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Finished. {dt_string}")